# Gemma 4 31B V8.0: All 60 Layers 6-Sense Player POV (See + Hear + Feel + Smell + Taste + Cuneiform S^7) + Spatial Semantic Districts + GTA LOD (G4 96GB / A100 80GB / L4 High-RAM Max VRAM)

**V8.0 6-Sense Player Point-of-Perspective (POV) Full-Model Architecture (`v5-qat-dev`):**
- **32D 6-Sense Player POV Cortex (`extract_6sense_pov_features` + `SixSensePovRouterNet`):**
  1. **SEE (`4D` Visual / Spatial Tile Geometry):** `rms_log`, `crest_norm`, `tile_cv`, `edge_ratio` (`tri_modal.rs::SeeSpatial`).
  2. **HEAR (`6D` Auditory / Walsh-Hadamard 4-Band Sequency Spectrum):** `band_low..high`, `hf_ratio_norm`, `spectral_entropy` (`tri_modal.rs::HearSpectrum`).
  3. **FEEL (`4D` Tactile / Proprioceptive Trajectory Mechanics & Surface Roughness):** `token_velocity`, `kinematic_jerk`, `anisotropy`, `surface_roughness`.
  4. **SMELL (`4D` Olfactory / Ambient Chemical & Outlier Volatility Trace):** `kurtosis_trace`, `signed_skew`, `top1pct_concentration`, `rare_trace_ratio`.
  5. **TASTE (`6D` Gustatory / 5-Receptor Channel Valence Profile):** `sweet_pos`, `bitter_neg`, `sour_sharp`, `salty_dense`, `umami_coherent`, `valence_balance`.
  6. **CUNEIFORM (`8D` Walsh-Hadamard $S^7$ Unit Radical Signature):** $g = \text{normalize}(\bar{u} H_8) \in S^7$.
- **All 56 Cold Middle MLPs (`L02..L57`):** Co-activation Spatial Districts (`Zone 0 Shared Core = 12 blocks = 3,072 neurons` + `12` semantic city districts `= 72 blocks = 18,432 neurons`, `1,536` neurons per district), Rotated-Basis **GTA LOD Residual Split** (`W_base 1b` Row-Major pinned + `ΔW_hi-res 2b` across `lod_shared.t16c` & `lod_d0..lod_d11.t16c` with native `T16C_COLBLOCK_BIT = 0x20` on `down_proj`).
- **Max-VRAM Acceleration (`V8_0_MAX_VRAM=1`):**
  - **G4-96GB-MAX:** Full GPU HBM residency, `batch_seq=384` (processes all `768` sequences in **2 giant Tensor-Core batches**!), `als_ev_toks=8192`.
  - **A100-80GB-MAX:** Full GPU HBM residency, `batch_seq=256` (3 batches), `als_ev_toks=8192`.
  - **L4-22GB-MAX:** Student HBM residency (`resident_stud_gpu=True`, zero student PCIe copies!), `batch_seq=96`, automatic Stage 2 3-Bit Vision checkpoint warm-start.

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/DannyB-bit/Language-U-Gemma4.git"
REPO_DIR = Path("/content/Language-U-Gemma4")
DRIVE_V80 = Path("/content/drive/MyDrive/gemma4_v8_0_player_pov")
DRIVE_V80.mkdir(parents=True, exist_ok=True)

if not REPO_DIR.is_dir():
    subprocess.run(["git", "clone", "-b", "v5-qat-dev", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", "v5-qat-dev"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", "origin/v5-qat-dev"], check=True)

print(subprocess.check_output(["git", "-C", str(REPO_DIR), "log", "-1", "--oneline"], text=True).strip(), flush=True)
print(subprocess.check_output(["nvidia-smi"], text=True), flush=True)

import torch
assert torch.cuda.is_available(), "CUDA GPU required! Switch Runtime -> Change runtime type -> A100 or G4 High-RAM."
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
smoke_t = (torch.ones((16, 16), device="cuda", dtype=torch.bfloat16) @ torch.ones((16, 16), device="cuda", dtype=torch.bfloat16)).sum().item()
print(f"[GPU Ready] {torch.cuda.get_device_name(0)} | VRAM={vram_gb:.2f} GiB | max_vram_mode=True | bf16_smoke={smoke_t:.1f}", flush=True)

_need_pip = False
try:
    import datasets, safetensors, tokenizers, huggingface_hub, PIL
    from transformers.models.gemma4.modeling_gemma4 import Gemma4VisionModel
except Exception:
    _need_pip = True
if _need_pip:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-U",
         "transformers>=4.51.0", "accelerate", "safetensors", "huggingface_hub", "datasets", "tokenizers", "pillow"],
        check=True,
    )

try:
    from google.colab import userdata
    hf_tok = userdata.get("HF_TOKEN")
    if hf_tok:
        os.environ["HF_TOKEN"] = hf_tok
        print("Loaded HF_TOKEN from Colab secrets.")
except Exception:
    pass

SCRIPT = REPO_DIR / "experiments/run270_rust/calib/31b/run_v8_0_player_pov_spatial_a100.py"
print("Running 6-gate V8.0 6-Sense Player POV CPU pre-flight selftest...", flush=True)
print(subprocess.check_output([sys.executable, str(SCRIPT), "--selftest"], text=True), flush=True)

In [ ]:
import json
import os
import subprocess
import sys
import time
from pathlib import Path

REPO_DIR = Path("/content/Language-U-Gemma4")
subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", "v5-qat-dev"], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", "origin/v5-qat-dev"], check=True)
SCRIPT = REPO_DIR / "experiments/run270_rust/calib/31b/run_v8_0_player_pov_spatial_a100.py"
LOG_PATH = Path("/content/v8_0_train.log")
DRIVE_V80 = Path("/content/drive/MyDrive/gemma4_v8_0_player_pov")

import torch
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
env = os.environ.copy()
env["V8_0_DRIVE_DIR"] = str(DRIVE_V80)
env["V8_0_OUT_DIR"] = "/content/v8_0_outputs"
env["V8_0_MAX_VRAM"] = "1"
env["V8_0_RESIDENT_GPU"] = "1" if vram_gb >= 64.0 else "0"
env["V8_0_RESIDENT_STUD_GPU"] = "1" if vram_gb >= 20.0 else "0"
env["V8_0_DISTRICT_DISTILL"] = "1"
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:512"
env["COLAB_AUTO_UNASSIGN"] = "1"
env["PYTHONUNBUFFERED"] = "1"

print(f"Launching ALL-60-LAYER V8.0 6-Sense Player POV + Spatial Districts + GTA LOD (Max VRAM {vram_gb:.1f} GiB) -> {LOG_PATH}...", flush=True)
with open(LOG_PATH, "w", encoding="utf-8") as lfp:
    proc = subprocess.Popen([sys.executable, "-u", str(SCRIPT), "--run"], stdout=lfp, stderr=subprocess.STDOUT, env=env)

seen = 0
while proc.poll() is None:
    time.sleep(15)
    if LOG_PATH.is_file():
        txt = LOG_PATH.read_text(encoding="utf-8", errors="replace")
        if len(txt) > seen:
            print(txt[seen:], end="", flush=True)
            seen = len(txt)

txt = LOG_PATH.read_text(encoding="utf-8", errors="replace")
if len(txt) > seen:
    print(txt[seen:], end="", flush=True)
assert proc.returncode == 0, f"V8.0 pipeline exited with code {proc.returncode}"

sys.path.insert(0, "/content/Language-U-Gemma4/experiments/run270_rust/calib/31b")
import run_v7_0_fullrank_crtr_gpu as v70
import spatial_district_converter as sdc
import train_cortical_router as crtr

rpt = json.loads((DRIVE_V80 / "v8_0_report.json").read_text(encoding="utf-8"))
nm_base, nb_base = v70.verify_t16c_integrity(DRIVE_V80 / "gemma4_31b_v8_0_base.t16c")
nm_sh, nb_sh = v70.verify_t16c_integrity(DRIVE_V80 / "gemma4_31b_v8_0_lod_shared.t16c")
d_ver = [v70.verify_t16c_integrity(DRIVE_V80 / f"gemma4_31b_v8_0_lod_d{d}.t16c") for d in range(12)]
cb_cnt = sdc.verify_t16c_file(DRIVE_V80 / "gemma4_31b_v8_0_lod_shared.t16c")["colblock_mats"]
cmeta, clayers = crtr.load_crtr(DRIVE_V80 / "gemma4_31b_v8_0_router.crtr")
print(f"[VERIFIED ALL V8.0 SHARDS] base_mats={nm_base} ({nb_base:,} B) shared_mats={nm_sh} (colblock={cb_cnt}) districts=12x{d_ver[0][0]} crtr_layers={len(clayers)}")
print(json.dumps({k: v for k, v in rpt.items() if k != 'layer_cos_summary'}, indent=2))

try:
    from google.colab import runtime
    print("[Auto-Unassign] Disconnecting Colab runtime...")
    runtime.unassign()
except Exception as exc:
    print(f"[Auto-Unassign Note] {exc}")